# 09 — Two Geometries on One Simplex

Close the geometry arc with the punchline that powers the rest of the course: the same simplex carries *two* distinct geometries — information and transport — and they give different answers to "what is the halfway distribution?"

**Prerequisites:** `02/06_the_probability_simplex`, `02/07_fisher_rao_metric`, `02/08_fisher_rao_geodesics`.

**What you'll be able to do**
- Distinguish a "vertical" (information) metric from a "horizontal" (transport) metric.
- Interpolate two distributions the *mixture* way and the *Wasserstein* way.
- Explain why the difference — bimodal blend vs sliding peak — is the reason the course turns to optimal transport.

You have built one full geometry of the simplex: Fisher–Rao (`02/07`–`02/08`). It is an *information* geometry — it compares masses bin by bin, ignoring what the bins *are*. This notebook reveals a second, completely different geometry hiding on the same simplex, and contrasting the two is the hinge into Modules 03 and 04.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.geometry.info_geometry import (
    mixture_interpolation, wasserstein_interpolation_1d,
)

np.random.seed(42)
viz.use_course_style()

## 1. A second geometry: optimal transport

Every way of comparing distributions we have used so far — KL and mutual information (divergences, not distances), and the Fisher–Rao distance — is *"vertical"*: it compares masses **bin by bin**, ignoring what the bins are. That is the right view when the bins are unrelated symbols (heads/tails, words in a vocabulary).

But often the bins **have their own geometry** — positions on a line, pixels in an image, neurons on a cortex — and *moving* mass between them has a cost. That is the **transport geometry** (Wasserstein) we will spend Modules 03 and 04 on. It gives a different distance, a different geodesic, and a different answer to "what is the natural path between two distributions?"

To see the contrast, take two Gaussian bumps on a 1-D line, well separated, and interpolate them in **two** geometries.

In [ ]:
support = np.linspace(0.0, 24.0, 200)

def bump(center, sigma=1.2):
    raw = np.exp(-0.5 * ((support - center) / sigma) ** 2)
    return raw / raw.sum()

p_left = bump(center=4.0)
p_right = bump(center=20.0)

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(support, p_left, color=viz.SOURCE_COLOR, lw=2, label="$p_0$ centered at $x=4$")
ax.plot(support, p_right, color=viz.TARGET_COLOR, lw=2, label="$p_1$ centered at $x=20$")
ax.set_xlabel("position  $x$")
ax.set_ylabel("probability mass")
ax.set_title("Two distributions on a line — same simplex, two metrics ahead", pad=12)
ax.legend()
plt.show()

**Read the figure.** Two well-separated bumps of the same width. We now ask: what is the "halfway distribution"? The two geometries will give two genuinely different answers.

## 2. Two interpolations — the punchline

For $t \in \{0, 0.25, 0.5, 0.75, 1\}$ we compute:

- the **mixture interpolation** (the *information* answer): $p_t(x) = (1-t)\,p_0(x) + t\,p_1(x)$ — a linear blend of masses, bin by bin;
- the **Wasserstein interpolation** (the *transport* answer): each unit of mass at quantile $u$ in $p_0$ slides to the quantile-$u$ position in $p_1$ (McCann, 1997).

Watch what each one does.

**The slide by hand, on four atoms.** Put half the mass at $x = 0$ and half at $x = 2$ for $p_0$, and a quarter at $x = 10$ and three quarters at $x = 14$ for $p_1$. Read each distribution as a staircase of quantile levels $u \in (0, 1]$ and cut the levels wherever either staircase steps — at $u = \tfrac14$ and $u = \tfrac12$:

- levels $(0, \tfrac14]$: mass $\tfrac14$ goes from $0$ to $10$, and at $t = \tfrac12$ it sits at $5$;
- levels $(\tfrac14, \tfrac12]$: mass $\tfrac14$ goes from $0$ to $14$, and sits at $7$;
- levels $(\tfrac12, 1]$: mass $\tfrac12$ goes from $2$ to $14$, and sits at $8$.

The transport midpoint is $\tfrac14\,\delta_5 + \tfrac14\,\delta_7 + \tfrac12\,\delta_8$, three atoms between the two piles. The mixture midpoint keeps all four atoms where they were, with half their mass: $\tfrac14$ at $0$ and $2$, $\tfrac18$ at $10$, $\tfrac38$ at $14$. Both midpoints have mean $7$, halfway between the means $1$ and $13$, so the mean cannot tell them apart; where the mass sits can. The cell below repeats the computation, and `03/06` states that this quantile matching is the optimal plan in one dimension and checks it against the linear program, while `03/07` turns the slide into McCann's geodesic. The helper used for the bumps does the same with a thousand equal-mass particles, rebinned on the grid.

In [ ]:
# The quantile slide of the four-atom example, computed exactly.
x0, m0 = np.array([0.0, 2.0]), np.array([0.5, 0.5])
x1, m1 = np.array([10.0, 14.0]), np.array([0.25, 0.75])
c0, c1 = np.cumsum(m0), np.cumsum(m1)
cuts = np.union1d(c0, c1)                      # quantile levels where either staircase steps
starts = np.concatenate([[0.0], cuts[:-1]])

t = 0.5
mid_x, mid_m = [], []
for a, b in zip(starts, cuts):
    u = 0.5 * (a + b)                          # any level inside the slice picks the same atoms
    src, dst = x0[np.searchsorted(c0, u)], x1[np.searchsorted(c1, u)]
    mid_x.append((1 - t) * src + t * dst)
    mid_m.append(b - a)
    print(f"levels ({a:.2f}, {b:.2f}]: mass {b - a:.2f} from x = {src:g} to x = {dst:g}, at t = 1/2 at x = {mid_x[-1]:g}")

mix_x = np.concatenate([x0, x1])
mix_m = np.concatenate([(1 - t) * m0, t * m1])
print("mixture midpoint:", ", ".join(f"{m:.3f} at {x:g}" for x, m in zip(mix_x, mix_m)))
print(f"means: p0 {np.dot(x0, m0):g}, p1 {np.dot(x1, m1):g}, transport midpoint {np.dot(mid_x, mid_m):g}, mixture midpoint {np.dot(mix_x, mix_m):g}")

In [ ]:
ts = [0.0, 0.25, 0.5, 0.75, 1.0]
mix_snaps = [mixture_interpolation(p_left, p_right, t) for t in ts]
w2_snaps = [wasserstein_interpolation_1d(p_left, p_right, support, t) for t in ts]

viz.plot_interpolation_panel(
    support,
    ts,
    {"mix": mix_snaps, "w2": w2_snaps},
    titles={
        "mix": "Mixture interpolation — amplitudes morph (bimodal at $t=0.5$)",
        "w2": "Wasserstein interpolation — the peak slides (single mode)",
    },
)
plt.show()

**Read both panels — this is the whole arc.**

- **Top (mixture / information).** At $t = 0.5$ the distribution is **bimodal**: half the mass at $x = 4$, half at $x = 20$, and *nothing in between*. The mixture does not care that there is empty space between the peaks; it blends *amplitudes*. The "halfway distribution" is a mixture, not a displacement.
- **Bottom (Wasserstein / transport).** The peak **slides** smoothly from $x = 4$ to $x = 20$, staying a single bump of nearly the same width. Wasserstein **knows** the bins live on a line and that empty space between two piles costs something to cross.

Same endpoints, same simplex — **two geometries, two different shortest paths**. The Fisher–Rao bow of `02/08` is the curved cousin of the mixture path; both belong to the **information** family. The Wasserstein interpolation belongs to a different family altogether, and that is the entire reason Modules 03 and 04 build optimal transport, then its quantum lift. The mixture path is a straight line of the *mixture* (m-) connection of information geometry; its dual, the *exponential* (e-) connection, draws straight lines in $\log p$, $p_t \propto p_0^{1-t}\,p_1^{t}$ (Amari 2016). Exercise 4 shows that for two Gaussian bumps of the same width this path slides too, and why that comes from the Gaussian shape rather than from the line. *(Cf. Peyré & Cuturi, Fig. 7.4.)*

## 3. Dictionary update

The geometry arc adds the connection row and returns to the metric row, which entered the dictionary in `02/07`. The quantum lift of the metric arrives in `02/13` (Bures); the Wasserstein geometry of this notebook earns its own rows in Module 03, and its quantum lift in Module 04.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| Fisher information metric on the simplex, $ds^2 = \sum_i dp_i^2/p_i$ | quantum Fisher information: a family of monotone metrics, the smallest being the Bures metric (`02/13`) |
| mixture (m-) vs exponential (e-) connections | quantum dual connections (not built in this course) |

## Your turn

1. **Bimodality threshold.** Move the right bump toward the left and find the separation at which the mixture midpoint stops being bimodal. How does it compare to $2\sigma$?
2. **Wasserstein stays unimodal.** Confirm the Wasserstein midpoint is a single bump for *any* separation. Why does sliding preserve the shape while blending does not?
3. **Same endpoints, different middle.** For your own pair of bumps, overlay the mixture and Wasserstein midpoints on one axis and describe the difference in a sentence.
4. **The other straight line.** Compute the e-midpoint $\sqrt{p_0\,p_1}$, normalised, for the two bumps and overlay it on the Wasserstein midpoint. Then repeat with two flat blocks that do not overlap. Why does the e-path slide in the first case and cease to exist in the second, while the Wasserstein path slides in both?

## What you built

- You distinguished a "vertical" information metric from a "horizontal" transport metric.
- You interpolated two distributions the mixture way (bimodal blend) and the Wasserstein way (sliding peak).
- You saw, in one figure, why the course turns to optimal transport — and added the geometry rows to the dictionary.

This is the conceptual hinge of the whole course. Everything from here — classical optimal transport (Module 03) and its quantum lift (Module 04) — develops the transport geometry you met here. Beautifully done.

## What's next

We close Module 02 by lifting the information toolbox to density matrices. In `02/10_umegaki_relative_entropy` the quantum KL reappears — and revives the $|+\rangle$ vs $I/2$ seed from `01/14`: it sees the coherence a classical, diagonal-only KL cannot.

## References

- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), ch. 7. DOI:10.1561/2200000073
- R. J. McCann, "A convexity principle for interacting gases", *Advances in Mathematics* 128, 153–179 (1997). DOI:10.1006/aima.1997.1634
- S.-i. Amari, *Information Geometry and Its Applications*, Springer (2016).

**Previous:** `notebooks/02_InformationAndGeometry/08_fisher_rao_geodesics.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/10_umegaki_relative_entropy.ipynb`